In [22]:
import boto3

# Assume into OrganizationAccountAccessRole in a target account
# By default this uses your *current* account ID; change TARGET_ACCOUNT_ID
# if you want to assume into a different AWS account.
base_session = boto3.session.Session()
sts = base_session.client("sts")
current_identity = sts.get_caller_identity()

TARGET_ACCOUNT_ID = 396511695522  # TODO: set to another account ID if needed
ROLE_NAME = "OrganizationAccountAccessRole"
ROLE_ARN = f"arn:aws:iam::{TARGET_ACCOUNT_ID}:role/{ROLE_NAME}"

print("Assuming role:", ROLE_ARN)
assumed = sts.assume_role(
    RoleArn=ROLE_ARN,
    RoleSessionName="rgasa-purge-session",
)
creds = assumed["Credentials"]

# Create a session with the assumed-role credentials
assumed_session = boto3.session.Session(
    aws_access_key_id=creds["AccessKeyId"],
    aws_secret_access_key=creds["SecretAccessKey"],
    aws_session_token=creds["SessionToken"],
    region_name=base_session.region_name or "eu-west-1",
)

# Optionally make the assumed role the default for future boto3.client/resource calls
boto3.setup_default_session(
    aws_access_key_id=creds["AccessKeyId"],
    aws_secret_access_key=creds["SecretAccessKey"],
    aws_session_token=creds["SessionToken"],
    region_name=base_session.region_name or "eu-west-1",
)

# Confirm who we are now
identity = assumed_session.client("sts").get_caller_identity()

print("AWS STS get_caller_identity() after assume-role:")
print(f"  Account:   {identity['Account']}")
print(f"  UserId:    {identity['UserId']}")
print(f"  ARN:       {identity['Arn']}")
print(f"  Region:    {assumed_session.region_name}")

print("You can now use boto3.client(...) and it will use the assumed role.")

Assuming role: arn:aws:iam::396511695522:role/OrganizationAccountAccessRole
AWS STS get_caller_identity() after assume-role:
  Account:   396511695522
  UserId:    AROAJ5YTGMWHKRNBYDR4Y:rgasa-purge-session
  ARN:       arn:aws:sts::396511695522:assumed-role/OrganizationAccountAccessRole/rgasa-purge-session
  Region:    eu-west-1
You can now use boto3.client(...) and it will use the assumed role.


In [ ]:
import boto3
from datetime import datetime, timezone

BUCKET_NAME = "comotion-comodash-rgasa-datainput"
START = datetime(2025, 12, 5, 14, 30, 0, tzinfo=timezone.utc)
END = datetime(2025, 12, 9, 23, 59, 59, tzinfo=timezone.utc)
# PREFIXES = ["inforce/", "terminations/", "treaty_info/"]
PREFIXES = ["terminations/"]

s3 = boto3.client("s3")

matched_objects = []

for prefix in PREFIXES:
    paginator = s3.get_paginator("list_objects_v2")
    for page in paginator.paginate(Bucket=BUCKET_NAME, Prefix=prefix):
        for obj in page.get("Contents", []):
            last_modified = obj["LastModified"]
            if START <= last_modified <= END:
                matched_objects.append({
                    "Key": obj["Key"],
                    "LastModified": last_modified,
                    "Size": obj["Size"],
                })

print(f"Found {len(matched_objects)} objects in {BUCKET_NAME} between {START} and {END} (UTC)")
for o in matched_objects:
    print(f"{o['LastModified'].isoformat()}\t{o['Size']}\t{o['Key']}")

Found 3532 objects in comotion-comodash-rgasa-datainput between 2025-12-05 14:30:00+00:00 and 2025-12-09 23:59:59+00:00 (UTC)
2025-12-08T15:10:16+00:00	680524	inforce/data_import_batch=2025-12-08/service_client_id=0/00242cf0-d448-11f0-94a9-af590a79eef3.csv.gz
2025-12-08T15:03:07+00:00	609616	inforce/data_import_batch=2025-12-08/service_client_id=0/006263e0-d447-11f0-8de2-bf1ec8265218.csv.gz
2025-12-08T14:55:59+00:00	667248	inforce/data_import_batch=2025-12-08/service_client_id=0/013539b0-d446-11f0-94a9-af590a79eef3.csv.gz
2025-12-08T14:48:50+00:00	595389	inforce/data_import_batch=2025-12-08/service_client_id=0/016b5a50-d445-11f0-94a9-af590a79eef3.csv.gz
2025-12-08T08:00:49+00:00	1820187	inforce/data_import_batch=2025-12-08/service_client_id=0/0198e9e0-d40c-11f0-b8b9-cb4756122fe6.csv.gz
2025-12-08T15:10:19+00:00	667929	inforce/data_import_batch=2025-12-08/service_client_id=0/01e7d5f0-d448-11f0-94a9-af590a79eef3.csv.gz
2025-12-08T15:03:11+00:00	592622	inforce/data_import_batch=2025-12-08

In [ ]:
import boto3
from datetime import datetime, timezone

BUCKET_NAME = "comotion-comodash-rgasa-datainput"
START = datetime(2025, 12, 5, 14, 30, 0, tzinfo=timezone.utc)
END = datetime(2025, 12, 9, 23, 59, 59, tzinfo=timezone.utc)
PREFIXES = ["inforce/", "terminations/", "treaty_info/"]

s3 = boto3.client("s3")

matched_objects = []

for prefix in PREFIXES:
    paginator = s3.get_paginator("list_objects_v2")
    for page in paginator.paginate(Bucket=BUCKET_NAME, Prefix=prefix):
        for obj in page.get("Contents", []):
            last_modified = obj["LastModified"]
            if START <= last_modified <= END:
                matched_objects.append({
                    "Key": obj["Key"],
                    "LastModified": last_modified,
                    "Size": obj["Size"],
                })

print(f"Found {len(matched_objects)} objects in {BUCKET_NAME} between {START} and {END} (UTC)")
for o in matched_objects:
    print(f"{o['LastModified'].isoformat()}\t{o['Size']}\t{o['Key']}")

Found 0 objects in comotion-comodash-rgasa-datainput between 2025-12-05 14:30:00+00:00 and 2025-12-09 23:59:59+00:00 (UTC)
